In [1]:
import pandas as pd
import requests
from PIL import Image
from io import BytesIO
import torch
from transformers import AutoProcessor, AutoModel
import numpy as np
import os
import time
import sys
from tqdm.auto import tqdm

# --- Configuration ---
# initial_filter_keyword = "doctor"
initial_filter_keyword = ""
directory = "dataset/relaion2B-en-research-safe"
parquet_file = "0000.parquet"
output_base_dir = "clip_embeddings_resumable"
clip_model_names = ["openai/clip-vit-base-patch16", "openai/clip-vit-base-patch32", "openai/clip-vit-large-patch14"]
model_index = 2
model_name = clip_model_names[model_index]

output_subdir_name = f"{initial_filter_keyword.replace(' ', '_')}_{model_name.replace('/', '_').replace('-', '_')}" if initial_filter_keyword else f"all_images_{model_name.replace('/', '_').replace('-', '_')}"
output_dir_for_embeddings = os.path.join(output_base_dir, output_subdir_name)
os.makedirs(output_dir_for_embeddings, exist_ok=True)

output_filepath = os.path.join(output_dir_for_embeddings, f"{parquet_file.replace('.parquet', '')}_embeddings.parquet")

# --- Initial Data Loading and Filtering ---
if 'df' not in locals() and 'df' not in globals():
    print(f"Loading DataFrame from {os.path.join(directory, parquet_file)}...")
    try:
        df = pd.read_parquet(os.path.join(directory, parquet_file))
        print(f"DataFrame loaded with {len(df)} rows.")
    except FileNotFoundError:
        print(f"Error: Input parquet file not found at {os.path.join(directory, parquet_file)}")
        sys.exit(1)

if not initial_filter_keyword:
    filtered_data = df.copy()
    print("No filter keyword provided. Processing all images.")
else:
    filtered_data = df[df["caption"].str.contains(initial_filter_keyword, case=False, na=False)].copy()
    print(f"Filtered data for keyword '{initial_filter_keyword}' has {len(filtered_data)} rows.")

filtered_data = filtered_data.head(40000)
print(f"Processing the first {len(filtered_data)} filtered images.")


# --- Resumption Logic: Load existing embeddings if available ---
existing_embeddings_df = pd.DataFrame() # Initialize an empty DataFrame
processed_image_indices = set() # To store indices of already processed images

if os.path.exists(output_filepath):
    try:
        existing_embeddings_df = pd.read_parquet(output_filepath)
        processed_image_indices = set(existing_embeddings_df['original_image_index'].unique())
        print(f"Resuming: Found {len(existing_embeddings_df)} existing embeddings in {output_filepath}.")
        print(f"Already processed {len(processed_image_indices)} unique image indices.")
    except Exception as e:
        print(f"Could not load existing embeddings from {output_filepath}: {e}")
        print("Starting fresh (existing file might be corrupted or empty).")
        existing_embeddings_df = pd.DataFrame()
        processed_image_indices = set()

images_to_process = filtered_data[~filtered_data.index.isin(processed_image_indices)].copy()

if images_to_process.empty:
    print("All images in the filtered set have already been processed. Nothing to do.")
    sys.exit(0)
else:
    print(f"Starting processing for {len(images_to_process)} new/remaining images.")


# --- CLIP Model Loading ---
processor = AutoProcessor.from_pretrained(model_name)
model = AutoModel.from_pretrained(model_name)

device = "cuda" if torch.cuda.is_available() else "cpu"
model.to(device)
print(f"Using CLIP model: {model_name} on device: {device}")

# --- Helper function to load image from URL ---
def load_image_from_url(url):
    try:
        response = requests.get(url, timeout=10)
        response.raise_for_status()
        image = Image.open(BytesIO(response.content)).convert("RGB")
        return image
    except requests.exceptions.RequestException as e:
        # print(f"Error fetching image from {url}: {e}")
        return None
    except Image.UnidentifiedImageError:
        # print(f"Error: Cannot identify image file from {url}. Skipping.")
        return None
    except Exception as e:
        # print(f"An unexpected error occurred loading image from {url}: {e}")
        return None

# --- Calculate Embeddings with tqdm progress bar ---
new_embedding_results = []
save_interval = 1000 # Save every N images processed

for count, (index, row) in tqdm(enumerate(images_to_process.iterrows()),
                                 total=len(images_to_process),
                                 desc="Calculating Embeddings"):
    image_url = row["url"]
    caption = row["caption"]
    similarity = None
    parquet_file_name = parquet_file

    image = load_image_from_url(image_url)

    if image:
        try:
            inputs = processor(images=image, return_tensors="pt").to(device)
            with torch.no_grad():
                image_features = model.get_image_features(**inputs)
            image_embedding = image_features / image_features.norm(p=2, dim=-1, keepdim=True)

            new_embedding_results.append({
                'original_image_index': index,
                'url': image_url,
                'caption': caption,
                'embeddings_result': image_embedding.cpu().squeeze().numpy(),
                'similarity': similarity,
                'parquet_file_name': parquet_file_name
            })

        except Exception as e:
            # tqdm.write(f"Error processing image {image_url} with CLIP: {e}. Marking as failed.")
            new_embedding_results.append({
                'original_image_index': index,
                'url': image_url,
                'caption': caption,
                'embeddings_result': None,
                'similarity': similarity,
                'parquet_file_name': parquet_file_name
            })
    else:
        # tqdm.write(f"Error: Image from {image_url} skipped (failed to load). Marking as failed.")
        new_embedding_results.append({
            'original_image_index': index,
            'url': image_url,
            'caption': caption,
            'embeddings_result': None,
            'similarity': similarity,
            'parquet_file_name': parquet_file_name
        })

    # --- Checkpoint: Save periodically ---
    # This block now correctly updates existing_embeddings_df
    if (count + 1) % save_interval == 0 or (count + 1) == len(images_to_process):

        # 1. Create a DataFrame from the new results since the last checkpoint
        df_new_batch = pd.DataFrame(new_embedding_results)

        # 2. Concatenate all previously processed embeddings with the new batch
        # IMPORTANT: existing_embeddings_df now holds the cumulative data
        combined_df = pd.concat([existing_embeddings_df, df_new_batch], ignore_index=True)

        # 3. Deduplicate based on the original image index (keep the latest entry if any duplicate)
        combined_df.drop_duplicates(subset=['original_image_index'], keep='last', inplace=True)
        
        test_path = os.path.join(output_dir_for_embeddings, f"{parquet_file.replace('.parquet', '')}_embeddings_{len(combined_df)}.parquet")
        combined_df.to_parquet(test_path, index=False)

        # 4. Save the full combined and deduplicated DataFrame to disk
        combined_df.to_parquet(output_filepath, index=False)
        tqdm.write(f"--- Checkpoint saved: {len(combined_df)} embeddings (including failures) written to {output_filepath} ---")
        
        # 5. CRITICAL FIX: Update existing_embeddings_df to be the currently saved state.
        # This ensures the next checkpoint calculation includes all processed items so far.
        existing_embeddings_df = combined_df.copy() 

        # 6. Reset new_embedding_results for the next batch if not at the end
        if count + 1 < len(images_to_process):
            new_embedding_results = []

# Final save (This block might be redundant if the last checkpoint saves everything, but good for safety)
# This handles cases where the loop finishes but the last batch wasn't big enough for a save_interval trigger
print("Finalizing remaining embeddings...")
if new_embedding_results:
    df_new_batch = pd.DataFrame(new_embedding_results)
    current_embeddings_df = pd.concat([existing_embeddings_df, df_new_batch], ignore_index=True)
    current_embeddings_df.drop_duplicates(subset=['original_image_index'], keep='last', inplace=True)

    os.makedirs(os.path.dirname(output_filepath.replace('clip_embeddings_resumable', 'final_clip_embeddings')), exist_ok=True)
    current_embeddings_df.to_parquet(output_filepath.replace('clip_embeddings_resumable', 'final_clip_embeddings'), index=False)
    tqdm.write(f"--- Final save: {len(current_embeddings_df)} embeddings (including failures) written to {output_filepath} ---")

print(f"\nImage embedding process complete for {model_name}.")
final_saved_df = pd.read_parquet(output_filepath)
final_successful_embeddings_count = len(final_saved_df.dropna(subset=['embeddings_result']))
print(f"Total unique *successful* embeddings saved: {final_successful_embeddings_count}")
print(f"Total unique *attempted* embeddings (including failures) saved: {len(final_saved_df)}")
print("\nSample of the final saved DataFrame (may include rows with None in embeddings_result):")
# print(final_saved_df.head())

Loading DataFrame from dataset/relaion2B-en-research-safe\0000.parquet...
DataFrame loaded with 16388209 rows.
No filter keyword provided. Processing all images.
Processing the first 40000 filtered images.
Resuming: Found 20000 existing embeddings in clip_embeddings_resumable\all_images_openai_clip_vit_large_patch14\0000_embeddings.parquet.
Already processed 20000 unique image indices.
Starting processing for 20000 new/remaining images.
Using CLIP model: openai/clip-vit-large-patch14 on device: cuda


Calculating Embeddings:   0%|          | 0/20000 [00:00<?, ?it/s]

c:\MastersRepos\ARI5902-Research-Topics-in-AI\LAION-5B Testing\.venv\lib\site-packages\transformers\models\clip\modeling_clip.py:491: UserWarning: 1Torch was not compiled with flash attention. (Triggered internally at C:\actions-runner\_work\pytorch\pytorch\builder\windows\pytorch\aten\src\ATen\native\transformers\cuda\sdp_utils.cpp:555.)
  attn_output = torch.nn.functional.scaled_dot_product_attention(
c:\MastersRepos\ARI5902-Research-Topics-in-AI\LAION-5B Testing\.venv\lib\site-packages\PIL\Image.py:1056: UserWarning: Palette images with Transparency expressed in bytes should be converted to RGBA images
  warnings.warn(


--- Checkpoint saved: 21000 embeddings (including failures) written to clip_embeddings_resumable\all_images_openai_clip_vit_large_patch14\0000_embeddings.parquet ---


KeyboardInterrupt: 

In [16]:
import pandas as pd
import os

# Recommended fix: Use 'r' before the string
file_path = r'C:\MastersRepos\ARI5902-Research-Topics-in-AI\LAION-5B Testing\clip_embeddings_resumable\all_images_openai_clip_vit_large_patch14\0000_embeddings.parquet'

# file_path = r'C:\MastersRepos\ARI5902-Research-Topics-in-AI\LAION-5B Testing\final_clip_embeddings\all_images_openai_clip_vit_large_patch14\0000_embeddings.parquet'

test = pd.read_parquet(file_path)
print("Parquet file loaded successfully using a raw string!")
print(test)

Parquet file loaded successfully using a raw string!
       original_image_index  \
0                         0   
1                         1   
2                         2   
3                         3   
4                         4   
...                     ...   
19995                 19995   
19996                 19996   
19997                 19997   
19998                 19998   
19999                 19999   

                                                     url  \
0      https://www.picclickimg.com/d/l400/pict/333262...   
1      https://nationalbookswap.com/pbs/m/42/0942/978...   
2      https://d1251d0o0760fi.cloudfront.net/catalog/...   
3      https://c8.alamy.com/comp/C84HAM/herd-of-cows-...   
4      https://ss-pics.s3.eu-west-1.amazonaws.com/fil...   
...                                                  ...   
19995  https://www.texasimpaireddrivingtaskforce.org/...   
19996  https://images.bigcartel.com/product_images/17...   
19997  http://cdn.shopify.com/s/fi

In [3]:
# Works but saving doesnt

# import pandas as pd
# import requests
# from PIL import Image
# from io import BytesIO
# import torch
# from transformers import AutoProcessor, AutoModel
# import numpy as np
# import os
# import sys
# import time
# from tqdm.auto import tqdm
# from concurrent.futures import ThreadPoolExecutor

# # --- Configuration ---
# initial_filter_keyword = ""
# directory = "dataset/relaion2B-en-research-safe"
# parquet_file = "0000.parquet"
# output_base_dir = "clip_embeddings_resumable"
# clip_model_names = ["openai/clip-vit-base-patch16", "openai/clip-vit-base-patch32", "openai/clip-vit-large-patch14"]
# model_index = 2
# model_name = clip_model_names[model_index]
# batch_size = 32
# save_interval = 1000
# first_x_rows = 40000

# # Output path setup
# output_subdir_name = f"{initial_filter_keyword.replace(' ', '_')}_{model_name.replace('/', '_').replace('-', '_')}" if initial_filter_keyword else f"all_images_{model_name.replace('/', '_').replace('-', '_')}"
# output_dir_for_embeddings = os.path.join(output_base_dir, output_subdir_name)
# os.makedirs(output_dir_for_embeddings, exist_ok=True)
# output_filepath = os.path.join(output_dir_for_embeddings, f"{parquet_file.replace('.parquet', '')}_embeddings.parquet")

# # --- Load Data ---
# try:
#     df = pd.read_parquet(os.path.join(directory, parquet_file))
#     print(f"Loaded DataFrame with {len(df)} rows.")
# except FileNotFoundError:
#     print(f"Parquet file not found at {os.path.join(directory, parquet_file)}")
#     sys.exit(1)

# if initial_filter_keyword:
#     filtered_data = df[df["caption"].str.contains(initial_filter_keyword, case=False, na=False)].copy()
#     print(f"Filtered for keyword '{initial_filter_keyword}', remaining: {len(filtered_data)} rows.")
# else:
#     filtered_data = df.copy()
#     print("No keyword filter applied. Using all data.")

# if first_x_rows:
#     print(f"Limiting to first {first_x_rows} rows for processing.")
#     filtered_data = filtered_data.head(first_x_rows)
# print(f"Processing first {len(filtered_data)} rows.")

# # --- Resume support ---
# existing_embeddings_df = pd.DataFrame()
# processed_image_indices = set()

# if os.path.exists(output_filepath):
#     try:
#         existing_embeddings_df = pd.read_parquet(output_filepath)
#         processed_image_indices = set(existing_embeddings_df['original_image_index'].unique())
#         print(f"Resuming: Found {len(existing_embeddings_df)} embeddings already saved.")
#     except Exception as e:
#         print(f"Warning: Could not load previous embeddings: {e}")

# images_to_process = filtered_data[~filtered_data.index.isin(processed_image_indices)].copy()
# if images_to_process.empty:
#     print("All images have already been processed.")
#     sys.exit(0)

# # --- Load CLIP model ---
# processor = AutoProcessor.from_pretrained(model_name)
# model = AutoModel.from_pretrained(model_name)
# device = "cuda" if torch.cuda.is_available() else "cpu"
# model.to(device)
# print(f"Model '{model_name}' loaded on {device}")

# # --- Image loading helper ---
# def fetch_image(row):
#     try:
#         response = requests.get(row["url"], timeout=10)
#         response.raise_for_status()
#         image = Image.open(BytesIO(response.content)).convert("RGB")
#         return (row.name, row["url"], row["caption"], image)
#     except:
#         return (row.name, row["url"], row["caption"], None)

# # --- Batch processing loop ---
# new_embedding_results = []
# rows = list(images_to_process.iterrows())

# for i in tqdm(range(0, len(rows), batch_size), desc="Calculating Embeddings in Batches"):
#     batch = rows[i:i+batch_size]

#     with ThreadPoolExecutor(max_workers=16) as executor:
#         image_data = list(executor.map(lambda x: fetch_image(x[1]), batch))

#     # Filter valid images
#     valid_batch = [(idx, url, cap, img) for idx, url, cap, img in image_data if img is not None]
#     if not valid_batch:
#         continue

#     indices, urls, captions, pil_images = zip(*valid_batch)

#     try:
#         inputs = processor(images=list(pil_images), return_tensors="pt", padding=True).to(device)
#         with torch.no_grad():
#             image_features = model.get_image_features(**inputs)
#         image_embeddings = image_features / image_features.norm(p=2, dim=-1, keepdim=True)

#         for j, embedding in enumerate(image_embeddings):
#             new_embedding_results.append({
#                 'original_image_index': indices[j],
#                 'url': urls[j],
#                 'caption': captions[j],
#                 'embeddings_result': embedding.cpu().numpy(),
#                 'similarity': None,
#                 'parquet_file_name': parquet_file
#             })

#     except Exception as e:
#         tqdm.write(f"Error during batch: {e}")
#         for idx, url, caption in zip(indices, urls, captions):
#             new_embedding_results.append({
#                 'original_image_index': idx,
#                 'url': url,
#                 'caption': caption,
#                 'embeddings_result': None,
#                 'similarity': None,
#                 'parquet_file_name': parquet_file
#             })

#     # --- Periodic Save ---
#     if (i + batch_size >= len(rows)) or ((i // batch_size + 1) * batch_size % save_interval == 0):
#         df_new_batch = pd.DataFrame(new_embedding_results)
#         combined_df = pd.concat([existing_embeddings_df, df_new_batch], ignore_index=True)
#         combined_df.drop_duplicates(subset=['original_image_index'], keep='last', inplace=True)
#         combined_df.to_parquet(output_filepath, index=False)
#         tqdm.write(f"Checkpoint saved with {len(combined_df)} embeddings.")
#         existing_embeddings_df = combined_df.copy()
#         new_embedding_results = []

# # --- Final Save ---
# if new_embedding_results:
#     df_new_batch = pd.DataFrame(new_embedding_results)
#     combined_df = pd.concat([existing_embeddings_df, df_new_batch], ignore_index=True)
#     combined_df.drop_duplicates(subset=['original_image_index'], keep='last', inplace=True)

#     final_path = output_filepath.replace('clip_embeddings_resumable', 'final_clip_embeddings')
#     os.makedirs(os.path.dirname(final_path), exist_ok=True)
#     combined_df.to_parquet(final_path, index=False)
#     print(f"\nFinal save: {len(combined_df)} embeddings written to {final_path}")

# # --- Completion ---
# print("\nImage embedding generation complete.")
# final_df = pd.read_parquet(output_filepath)
# successful = len(final_df.dropna(subset=['embeddings_result']))
# print(f"✅ Successful embeddings: {successful}")
# print(f"📦 Total attempted (including failed): {len(final_df)}")


In [1]:
import pandas as pd
import requests
from PIL import Image
from io import BytesIO
import torch
from transformers import AutoProcessor, AutoModel
import numpy as np
import os
import time
import sys
from tqdm.auto import tqdm
from concurrent.futures import ThreadPoolExecutor

# --- Configuration ---
initial_filter_keyword = ""
directory = "dataset/relaion2B-en-research-safe"
parquet_file = "0000.parquet"
output_base_dir = "clip_embeddings_resumable"
clip_model_names = ["openai/clip-vit-base-patch16", "openai/clip-vit-base-patch32", "openai/clip-vit-large-patch14"]
model_index = 2
model_name = clip_model_names[model_index]
batch_size = number_of_workers = 64
save_interval = 1000

output_subdir_name = f"{initial_filter_keyword.replace(' ', '_')}_{model_name.replace('/', '_').replace('-', '_')}" if initial_filter_keyword else f"all_images_{model_name.replace('/', '_').replace('-', '_')}"
output_dir_for_embeddings = os.path.join(output_base_dir, output_subdir_name)
os.makedirs(output_dir_for_embeddings, exist_ok=True)
output_filepath = os.path.join(output_dir_for_embeddings, f"{parquet_file.replace('.parquet', '')}_embeddings.parquet")

# --- Load input data ---
print(f"Loading DataFrame from {os.path.join(directory, parquet_file)}...")
try:
    df = pd.read_parquet(os.path.join(directory, parquet_file))
    print(f"DataFrame loaded with {len(df)} rows.")
except FileNotFoundError:
    print(f"Error: Input parquet file not found at {os.path.join(directory, parquet_file)}")
    sys.exit(1)

filtered_data = df.copy() if not initial_filter_keyword else df[df["caption"].str.contains(initial_filter_keyword, case=False, na=False)].copy()
print(f"Filtered data: {len(filtered_data)} rows. Limiting to first 60,000.")
filtered_data = filtered_data.head(60000)

# --- Resumption Logic ---
existing_embeddings_df = pd.DataFrame()
processed_image_indices = set()

if os.path.exists(output_filepath):
    try:
        existing_embeddings_df = pd.read_parquet(output_filepath)
        processed_image_indices = set(existing_embeddings_df['original_image_index'].unique())
        print(f"Resuming: Found {len(existing_embeddings_df)} existing embeddings.")
    except Exception as e:
        print(f"Warning loading existing embeddings: {e}")

images_to_process = filtered_data[~filtered_data.index.isin(processed_image_indices)].copy()
if images_to_process.empty:
    print("All images have been processed. Exiting.")
    sys.exit(0)
else:
    print(f"Remaining images to process: {len(images_to_process)}")

# --- Load CLIP model ---
processor = AutoProcessor.from_pretrained(model_name)
model = AutoModel.from_pretrained(model_name)
device = "cuda" if torch.cuda.is_available() else "cpu"
model.to(device)
print(f"Using CLIP model: {model_name} on device: {device}")

# --- Image fetch function ---
def fetch_image(index_row_tuple):
    index, row = index_row_tuple
    image_url = row["url"]
    caption = row["caption"]

    # start_time = time.time()

    try:
        response = requests.get(image_url, timeout=10)
        response.raise_for_status()
        image = Image.open(BytesIO(response.content)).convert("RGB")

        # duration = time.time() - start_time
        # print(f"[Downloaded] {image_url} in {duration:.2f}s")

        return (index, image_url, caption, image)
    except:
        return (index, image_url, caption, None)

# --- Main processing ---
new_embedding_results = []
total_processed = len(existing_embeddings_df)

for batch_start in tqdm(range(0, len(images_to_process), batch_size), desc="Processing Batches"):
    batch_end = min(batch_start + batch_size, len(images_to_process))
    batch_rows = list(images_to_process.iloc[batch_start:batch_end].iterrows())

    # Parallel image download
    st_time = time.time()
    with ThreadPoolExecutor(max_workers=number_of_workers) as executor:
        fetched_images = list(executor.map(fetch_image, batch_rows))
    elapsed_time = time.time() - st_time
    tqdm.write(f"Batch {batch_start // batch_size + 1} processed in {elapsed_time:.2f}s")

    valid_batch = [item for item in fetched_images if item[3] is not None]
    failed_batch = [item for item in fetched_images if item[3] is None]

    # Process valid images on GPU
    if valid_batch:
        indices, urls, captions, images = zip(*valid_batch)
        try:
            inputs = processor(images=list(images), return_tensors="pt", padding=True).to(device)
            with torch.no_grad():
                image_features = model.get_image_features(**inputs)
            image_embeddings = image_features / image_features.norm(p=2, dim=-1, keepdim=True)

            for i in range(len(valid_batch)):
                new_embedding_results.append({
                    'original_image_index': indices[i],
                    'url': urls[i],
                    'caption': captions[i],
                    'embeddings_result': image_embeddings[i].cpu().numpy(),
                    'similarity': None,
                    'parquet_file_name': parquet_file
                })
        except Exception as e:
            tqdm.write(f"⚠️ Error during GPU batch processing: {e}")
            for i in range(len(valid_batch)):
                new_embedding_results.append({
                    'original_image_index': indices[i],
                    'url': urls[i],
                    'caption': captions[i],
                    'embeddings_result': None,
                    'similarity': None,
                    'parquet_file_name': parquet_file
                })

    # Log failed downloads
    for idx, url, cap, _ in failed_batch:
        new_embedding_results.append({
            'original_image_index': idx,
            'url': url,
            'caption': cap,
            'embeddings_result': None,
            'similarity': None,
            'parquet_file_name': parquet_file
        })

    total_processed += len(fetched_images)

    # Checkpoint saving
    if total_processed % save_interval < batch_size or batch_end == len(images_to_process):
        df_new_batch = pd.DataFrame(new_embedding_results)
        combined_df = pd.concat([existing_embeddings_df, df_new_batch], ignore_index=True)
        combined_df.drop_duplicates(subset=['original_image_index'], keep='last', inplace=True)

        test_path = os.path.join(output_dir_for_embeddings, f"{parquet_file.replace('.parquet', '')}_embeddings_{len(combined_df)}.parquet")
        combined_df.to_parquet(test_path, index=False)
        combined_df.to_parquet(output_filepath, index=False)
        tqdm.write(f"✅ Checkpoint saved: {len(combined_df)} rows (including failures)")

        existing_embeddings_df = combined_df.copy()
        new_embedding_results = []

# --- Final Save ---
print("Finalizing remaining embeddings...")
if new_embedding_results:
    df_final = pd.DataFrame(new_embedding_results)
    combined_df = pd.concat([existing_embeddings_df, df_final], ignore_index=True)
    combined_df.drop_duplicates(subset=['original_image_index'], keep='last', inplace=True)

    final_dir = output_filepath.replace('clip_embeddings_resumable', 'final_clip_embeddings')
    os.makedirs(os.path.dirname(final_dir), exist_ok=True)
    combined_df.to_parquet(final_dir, index=False)

    tqdm.write(f"✅ Final save complete: {len(combined_df)} rows written to {final_dir}")

# --- Final Stats ---
print(f"\nEmbedding process complete for {model_name}")
final_df = pd.read_parquet(output_filepath)
success_count = len(final_df.dropna(subset=['embeddings_result']))
print(f"✅ Successful embeddings: {success_count}")
print(f"📦 Total attempted (incl. failed): {len(final_df)}")


Loading DataFrame from dataset/relaion2B-en-research-safe\0000.parquet...
DataFrame loaded with 16388209 rows.
Filtered data: 16388209 rows. Limiting to first 60,000.
Resuming: Found 40000 existing embeddings.
Remaining images to process: 20000
Using CLIP model: openai/clip-vit-large-patch14 on device: cuda


Processing Batches:   0%|          | 0/313 [00:00<?, ?it/s]

Batch 1 processed in 10.21s


c:\MastersRepos\ARI5902-Research-Topics-in-AI\LAION-5B Testing\.venv\lib\site-packages\transformers\models\clip\modeling_clip.py:491: UserWarning: 1Torch was not compiled with flash attention. (Triggered internally at C:\actions-runner\_work\pytorch\pytorch\builder\windows\pytorch\aten\src\ATen\native\transformers\cuda\sdp_utils.cpp:555.)
  attn_output = torch.nn.functional.scaled_dot_product_attention(


Batch 2 processed in 1.55s
Batch 3 processed in 10.22s
Batch 4 processed in 10.06s
Batch 5 processed in 10.31s
Batch 6 processed in 10.18s
Batch 7 processed in 10.25s


...

Batch 127 processed in 1.96s
⚠️ Error during GPU batch processing: mean must have 1 elements if it is an iterable, got 3
Batch 128 processed in 1.63s
Batch 129 processed in 10.20s
Batch 130 processed in 2.17s
Batch 131 processed in 10.09s
Batch 132 processed in 2.61s
Batch 133 processed in 10.50s
Batch 134 processed in 10.27s
Batch 135 processed in 2.65s
Batch 136 processed in 2.40s
Batch 137 processed in 4.62s
Batch 138 processed in 10.08s
Batch 139 processed in 11.17s
Batch 140 processed in 10.11s
Batch 141 processed in 10.08s
✅ Checkpoint saved: 49024 rows (including failures)
Batch 142 processed in 2.80s
Batch 143 processed in 1.56s
Batch 144 processed in 5.61s
Batch 145 processed in 10.15s
Batch 146 processed in 10.25s
Batch 147 processed in 10.99s
Batch 148 processed in 2.47s
Batch 149 processed in 3.61s
Batch 150 processed in 1.70s
Batch 151 processed in 11.15s
Batch 152 processed in 10.18s
Batch 153 processed in 10.04s
Batch 154 processed in 10.06s
Batch 155 processed in 10.36s

c:\MastersRepos\ARI5902-Research-Topics-in-AI\LAION-5B Testing\.venv\lib\site-packages\PIL\TiffImagePlugin.py:900: UserWarning: Corrupt EXIF data.  Expecting to read 4 bytes but only got 0. 
  warnings.warn(str(msg))


Batch 220 processed in 11.17s
Batch 221 processed in 10.09s
Batch 222 processed in 11.26s
Batch 223 processed in 10.26s
Batch 224 processed in 10.50s
Batch 225 processed in 2.37s
Batch 226 processed in 11.28s
Batch 227 processed in 11.16s
Batch 228 processed in 2.01s
Batch 229 processed in 10.24s
Batch 230 processed in 10.05s
Batch 231 processed in 10.49s
Batch 232 processed in 2.43s
Batch 233 processed in 10.23s
Batch 234 processed in 10.21s
Batch 235 processed in 13.74s
✅ Checkpoint saved: 55040 rows (including failures)
Batch 236 processed in 5.78s
Batch 237 processed in 20.19s
Batch 238 processed in 12.49s
Batch 239 processed in 11.16s
Batch 240 processed in 5.58s
Batch 241 processed in 2.28s
Batch 242 processed in 3.15s
Batch 243 processed in 10.12s
Batch 244 processed in 10.19s
Batch 245 processed in 10.81s
Batch 246 processed in 10.13s
Batch 247 processed in 11.21s
Batch 248 processed in 3.62s
Batch 249 processed in 10.21s
Batch 250 processed in 10.29s
✅ Checkpoint saved: 56000 

The channel dimension is ambiguous. Got image shape (1, 1, 3). Assuming channels are the first dimension.


Batch 303 processed in 10.09s
⚠️ Error during GPU batch processing: mean must have 1 elements if it is an iterable, got 3
Batch 304 processed in 11.29s
Batch 305 processed in 10.62s
Batch 306 processed in 2.16s
Batch 307 processed in 10.38s
Batch 308 processed in 70.89s
Batch 309 processed in 10.06s
Batch 310 processed in 10.26s
Batch 311 processed in 10.34s
Batch 312 processed in 2.30s
Batch 313 processed in 3.54s
✅ Checkpoint saved: 60000 rows (including failures)
Finalizing remaining embeddings...

Embedding process complete for openai/clip-vit-large-patch14
✅ Successful embeddings: 39902
📦 Total attempted (incl. failed): 60000


In [ ]:
# https://huggingface.co/datasets/laion/relaion2B-en-research-safe

In [1]:
# testing out larger batch size

import pandas as pd
import requests
from PIL import Image
from io import BytesIO
import torch
from transformers import AutoProcessor, AutoModel
import numpy as np
import os
import time
import sys
from tqdm.auto import tqdm
from concurrent.futures import ThreadPoolExecutor
import pyarrow as pa
import pyarrow.parquet as pq
import glob

# --- Configuration ---
initial_filter_keyword = ""
directory = "dataset/relaion2B-en-research-safe"
parquet_file = "0000.parquet"
output_base_dir = "clip_embeddings_resumable"
clip_model_names = ["openai/clip-vit-base-patch16", "openai/clip-vit-base-patch32", "openai/clip-vit-large-patch14"]
model_index = 2
model_name = clip_model_names[model_index]
batch_size = number_of_workers = 200
save_interval = 5000
row_grp_size = 300000
keep_n_recent_saves = 2

output_subdir_name = f"{initial_filter_keyword.replace(' ', '_')}_{model_name.replace('/', '_').replace('-', '_')}" if initial_filter_keyword else f"all_images_{model_name.replace('/', '_').replace('-', '_')}"
output_dir_for_embeddings = os.path.join(output_base_dir, output_subdir_name)
os.makedirs(output_dir_for_embeddings, exist_ok=True)
output_filepath = os.path.join(output_dir_for_embeddings, f"{parquet_file.replace('.parquet', '')}_embeddings.parquet")

# --- Load input data ---
print(f"Loading DataFrame from {os.path.join(directory, parquet_file)}...")
try:
    df = pd.read_parquet(os.path.join(directory, parquet_file))
    print(f"DataFrame loaded with {len(df)} rows.")
except FileNotFoundError:
    print(f"Error: Input parquet file not found at {os.path.join(directory, parquet_file)}")
    sys.exit(1)

filtered_data = df.copy() if not initial_filter_keyword else df[df["caption"].str.contains(initial_filter_keyword, case=False, na=False)].copy()

# df no longer required, removing for memory issues
del df

print(f"Filtered data: {len(filtered_data)} rows. Limiting to first 2,500,000.")
filtered_data = filtered_data.head(2500000)

# --- Resumption Logic ---
existing_embeddings_df = pd.DataFrame()
processed_image_indices = set()

if os.path.exists(output_filepath):
    try:
        existing_embeddings_df = pd.read_parquet(output_filepath)
        processed_image_indices = set(existing_embeddings_df['original_image_index'].unique())
        print(f"Resuming: Found {len(existing_embeddings_df)} existing embeddings.")
    except Exception as e:
        print(f"Warning loading existing embeddings: {e}")

images_to_process = filtered_data[~filtered_data.index.isin(processed_image_indices)].copy()
if images_to_process.empty:
    print("All images have been processed. Exiting.")
    sys.exit(0)
else:
    print(f"Remaining images to process: {len(images_to_process)}")

# --- Load CLIP model ---
processor = AutoProcessor.from_pretrained(model_name)
model = AutoModel.from_pretrained(model_name)
device = "cuda" if torch.cuda.is_available() else "cpu"
model.to(device)
print(f"Using CLIP model: {model_name} on device: {device}")

# --- Image fetch function ---
def fetch_image(index_row_tuple):
    index, row = index_row_tuple
    image_url = row["url"]
    caption = row["caption"]

    # start_time = time.time()

    try:
        response = requests.get(image_url, timeout=10)
        response.raise_for_status()
        image = Image.open(BytesIO(response.content)).convert("RGB")

        # duration = time.time() - start_time
        # print(f"[Downloaded] {image_url} in {duration:.2f}s")

        return (index, image_url, caption, image)
    except:
        return (index, image_url, caption, None)

# --- Main processing ---
new_embedding_results = []
total_processed = len(existing_embeddings_df)

for batch_start in tqdm(range(0, len(images_to_process), batch_size), desc="Processing Batches"):
    batch_end = min(batch_start + batch_size, len(images_to_process))
    batch_rows = list(images_to_process.iloc[batch_start:batch_end].iterrows())

    # Parallel image download
    st_time = time.time()
    with ThreadPoolExecutor(max_workers=number_of_workers) as executor:
        fetched_images = list(executor.map(fetch_image, batch_rows))
    elapsed_time = time.time() - st_time
    tqdm.write(f"Batch {batch_start // batch_size + 1} processed in {elapsed_time:.2f}s")

    valid_batch = [item for item in fetched_images if item[3] is not None]
    failed_batch = [item for item in fetched_images if item[3] is None]

    # Process valid images on GPU
    if valid_batch:
        indices, urls, captions, images = zip(*valid_batch)
        try:
            inputs = processor(images=list(images), return_tensors="pt", padding=True).to(device)
            with torch.no_grad():
                image_features = model.get_image_features(**inputs)
            image_embeddings = image_features / image_features.norm(p=2, dim=-1, keepdim=True)

            for i in range(len(valid_batch)):
                new_embedding_results.append({
                    'original_image_index': indices[i],
                    'url': urls[i],
                    'caption': captions[i],
                    'embeddings_result': image_embeddings[i].cpu().numpy(),
                    'similarity': None,
                    'parquet_file_name': parquet_file
                })
        except Exception as e:
            tqdm.write(f"⚠️ Error during GPU batch processing: {e}")
            for i in range(len(valid_batch)):
                new_embedding_results.append({
                    'original_image_index': indices[i],
                    'url': urls[i],
                    'caption': captions[i],
                    'embeddings_result': None,
                    'similarity': None,
                    'parquet_file_name': parquet_file
                })

    # Log failed downloads
    for idx, url, cap, _ in failed_batch:
        new_embedding_results.append({
            'original_image_index': idx,
            'url': url,
            'caption': cap,
            'embeddings_result': None,
            'similarity': None,
            'parquet_file_name': parquet_file
        })

    total_processed += len(fetched_images)

    # Checkpoint saving
    if total_processed % save_interval < batch_size or batch_end == len(images_to_process):
        df_new_batch = pd.DataFrame(new_embedding_results)
        combined_df = pd.concat([existing_embeddings_df, df_new_batch], ignore_index=True)
        combined_df.drop_duplicates(subset=['original_image_index'], keep='last', inplace=True)

        # test_path = os.path.join(output_dir_for_embeddings, f"{parquet_file.replace('.parquet', '')}_embeddings_{len(combined_df)}.parquet")
        # combined_df.to_parquet(test_path, index=False)
        # combined_df.to_parquet(output_filepath, index=False)

        table = pa.Table.from_pandas(combined_df)
        test_path = os.path.join(output_dir_for_embeddings, f"{parquet_file.replace('.parquet', '')}_embeddings_{len(combined_df)}.parquet")
        pq.write_table(table, test_path, row_group_size=row_grp_size)
        pq.write_table(table, output_filepath, row_group_size=row_grp_size)

        # --- Keep only the X most recent test parquet files ---
        # List all checkpoint files for this image set
        test_files = sorted(
            glob.glob(os.path.join(output_dir_for_embeddings, f"{parquet_file.replace('.parquet', '')}_embeddings_*.parquet")),
            key=os.path.getmtime,  # sort by modification time
            reverse=True
        )

        # Keep only the 2 most recent files
        for old_file in test_files[keep_n_recent_saves:]:
            try:
                os.remove(old_file)
                tqdm.write(f"🗑️ Deleted old checkpoint: {old_file}")
            except Exception as e:
                tqdm.write(f"⚠️ Could not delete {old_file}: {e}")

                tqdm.write(f"✅ Checkpoint saved: {len(combined_df)} rows (including failures)")

                existing_embeddings_df = combined_df.copy()
                new_embedding_results = []

# --- Final Save ---
print("Finalizing remaining embeddings...")
if new_embedding_results:
    df_final = pd.DataFrame(new_embedding_results)
    combined_df = pd.concat([existing_embeddings_df, df_final], ignore_index=True)
    combined_df.drop_duplicates(subset=['original_image_index'], keep='last', inplace=True)

    final_dir = output_filepath.replace('clip_embeddings_resumable', 'final_clip_embeddings')
    os.makedirs(os.path.dirname(final_dir), exist_ok=True)
    # combined_df.to_parquet(final_dir, index=False)
    table = pa.Table.from_pandas(combined_df)
    pq.write_table(table, final_dir, row_group_size=row_grp_size)

    tqdm.write(f"✅ Final save complete: {len(combined_df)} rows written to {final_dir}")

# --- Final Stats ---
print(f"\nEmbedding process complete for {model_name}")
final_df = pd.read_parquet(output_filepath)
success_count = len(final_df.dropna(subset=['embeddings_result']))
print(f"✅ Successful embeddings: {success_count}")
print(f"📦 Total attempted (incl. failed): {len(final_df)}")


Loading DataFrame from dataset/relaion2B-en-research-safe\0000.parquet...
DataFrame loaded with 16388209 rows.
Filtered data: 16388209 rows. Limiting to first 2,500,000.
Resuming: Found 1440000 existing embeddings.
Remaining images to process: 1060000
Using CLIP model: openai/clip-vit-large-patch14 on device: cuda


Processing Batches:   0%|          | 0/5300 [00:00<?, ?it/s]

Batch 1 processed in 20.25s


c:\MastersRepos\ARI5902-Research-Topics-in-AI\LAION-5B Testing\.venv\lib\site-packages\transformers\models\clip\modeling_clip.py:491: UserWarning: 1Torch was not compiled with flash attention. (Triggered internally at C:\actions-runner\_work\pytorch\pytorch\builder\windows\pytorch\aten\src\ATen\native\transformers\cuda\sdp_utils.cpp:555.)
  attn_output = torch.nn.functional.scaled_dot_product_attention(
c:\MastersRepos\ARI5902-Research-Topics-in-AI\LAION-5B Testing\.venv\lib\site-packages\PIL\Image.py:1056: UserWarning: Palette images with Transparency expressed in bytes should be converted to RGBA images
  warnings.warn(


Batch 2 processed in 11.12s
Batch 3 processed in 6.78s
Batch 4 processed in 10.99s
Batch 5 processed in 21.15s
Batch 6 processed in 10.19s
Batch 7 processed in 20.12s
Batch 8 processed in 11.55s
Batch 9 processed in 10.43s
Batch 10 processed in 21.60s
Batch 11 processed in 20.57s
Batch 12 processed in 10.32s
Batch 13 processed in 22.75s
Batch 14 processed in 5.77s
Batch 15 processed in 10.63s
Batch 16 processed in 13.15s
Batch 17 processed in 11.45s
Batch 18 processed in 10.39s
Batch 19 processed in 10.28s
Batch 20 processed in 10.68s
Batch 21 processed in 20.32s
Batch 22 processed in 10.20s
Batch 23 processed in 14.07s
Batch 24 processed in 10.49s
Batch 25 processed in 10.36s
🗑️ Deleted old checkpoint: clip_embeddings_resumable\all_images_openai_clip_vit_large_patch14\0000_embeddings_1435000.parquet
Batch 26 processed in 10.66s
Batch 27 processed in 10.57s
Batch 28 processed in 11.30s
Batch 29 processed in 10.41s
Batch 30 processed in 10.72s
Batch 31 processed in 14.61s
Batch 32 proce

...

The channel dimension is ambiguous. Got image shape (1, 1, 3). Assuming channels are the first dimension.


Batch 1775 processed in 11.42s
⚠️ Error during GPU batch processing: mean must have 1 elements if it is an iterable, got 3
🗑️ Deleted old checkpoint: clip_embeddings_resumable\all_images_openai_clip_vit_large_patch14\0000_embeddings_1785000.parquet
Batch 1776 processed in 10.78s
Batch 1777 processed in 10.35s
Batch 1778 processed in 10.28s
Batch 1779 processed in 10.72s
Batch 1780 processed in 20.10s
Batch 1781 processed in 20.31s
Batch 1782 processed in 11.17s
Batch 1783 processed in 10.23s
Batch 1784 processed in 10.50s
Batch 1785 processed in 10.43s
Batch 1786 processed in 10.32s
Batch 1787 processed in 11.36s
Batch 1788 processed in 11.20s
Batch 1789 processed in 11.28s
Batch 1790 processed in 20.23s
Batch 1791 processed in 10.27s
Batch 1792 processed in 10.45s
Batch 1793 processed in 10.40s
Batch 1794 processed in 10.66s
Batch 1795 processed in 15.00s
Batch 1796 processed in 11.24s
Batch 1797 processed in 7.74s
Batch 1798 processed in 10.63s
Batch 1799 processed in 10.50s
Batch 18

c:\MastersRepos\ARI5902-Research-Topics-in-AI\LAION-5B Testing\.venv\lib\site-packages\PIL\TiffImagePlugin.py:900: UserWarning: Corrupt EXIF data.  Expecting to read 4 bytes but only got 0. 
  warnings.warn(str(msg))


Batch 1858 processed in 10.33s
Batch 1859 processed in 14.10s
Batch 1860 processed in 30.81s
Batch 1861 processed in 11.25s
Batch 1862 processed in 10.48s
Batch 1863 processed in 10.80s
Batch 1864 processed in 20.30s
Batch 1865 processed in 11.23s
Batch 1866 processed in 11.39s
Batch 1867 processed in 11.39s
Batch 1868 processed in 10.56s
Batch 1869 processed in 4.53s
Batch 1870 processed in 20.39s
Batch 1871 processed in 10.37s
Batch 1872 processed in 10.29s
Batch 1873 processed in 10.42s
Batch 1874 processed in 11.33s
Batch 1875 processed in 10.55s
🗑️ Deleted old checkpoint: clip_embeddings_resumable\all_images_openai_clip_vit_large_patch14\0000_embeddings_1805000.parquet
Batch 1876 processed in 11.35s
Batch 1877 processed in 10.49s
Batch 1878 processed in 10.64s
Batch 1879 processed in 10.11s
Batch 1880 processed in 11.55s
Batch 1881 processed in 20.21s
Batch 1882 processed in 31.12s
Batch 1883 processed in 11.36s
Batch 1884 processed in 11.35s
Batch 1885 processed in 10.45s
Batch 1

KeyboardInterrupt: 

In [ ]:
# testing out larger batch size

import pandas as pd
import requests
from PIL import Image
from io import BytesIO
import torch
from transformers import AutoProcessor, AutoModel
import numpy as np
import os
import time
import sys
from tqdm.auto import tqdm
from concurrent.futures import ThreadPoolExecutor
import pyarrow as pa
import pyarrow.parquet as pq
import glob

# --- Configuration ---
initial_filter_keyword = ""
directory = "dataset/relaion2B-en-research-safe"
parquet_file = "0000.parquet"
output_base_dir = "clip_embeddings_resumable"
clip_model_names = ["openai/clip-vit-base-patch16", "openai/clip-vit-base-patch32", "openai/clip-vit-large-patch14"]
model_index = 2
model_name = clip_model_names[model_index]
batch_size = number_of_workers = 200
save_interval = 5000
row_grp_size = 300000
keep_n_recent_saves = 2
image_limit = 3_000_000

output_subdir_name = f"{initial_filter_keyword.replace(' ', '_')}_{model_name.replace('/', '_').replace('-', '_')}" if initial_filter_keyword else f"all_images_{model_name.replace('/', '_').replace('-', '_')}"
output_dir_for_embeddings = os.path.join(output_base_dir, output_subdir_name)
os.makedirs(output_dir_for_embeddings, exist_ok=True)
output_filepath = os.path.join(output_dir_for_embeddings, f"{parquet_file.replace('.parquet', '')}_embeddings.parquet")

# --- Load input data ---
print(f"Loading DataFrame from {os.path.join(directory, parquet_file)}...")
try:
    df = pd.read_parquet(os.path.join(directory, parquet_file))
    print(f"DataFrame loaded with {len(df)} rows.")
except FileNotFoundError:
    print(f"Error: Input parquet file not found at {os.path.join(directory, parquet_file)}")
    sys.exit(1)

filtered_data = df.copy() if not initial_filter_keyword else df[df["caption"].str.contains(initial_filter_keyword, case=False, na=False)].copy()

# df no longer required, removing for memory issues
del df

print(f"Filtered data: {len(filtered_data)} rows. Limiting to first {image_limit}.")
filtered_data = filtered_data.head(image_limit)

# --- Resumption Logic ---
existing_embeddings_df = pd.DataFrame()
processed_image_indices = set()

if os.path.exists(output_filepath):
    try:
        existing_embeddings_df = pd.read_parquet(output_filepath)
        processed_image_indices = set(existing_embeddings_df['original_image_index'].unique())
        print(f"Resuming: Found {len(existing_embeddings_df)} existing embeddings.")
    except Exception as e:
        print(f"Warning loading existing embeddings: {e}")

images_to_process = filtered_data[~filtered_data.index.isin(processed_image_indices)].copy()
if images_to_process.empty:
    print("All images have been processed. Exiting.")
    sys.exit(0)
else:
    print(f"Remaining images to process: {len(images_to_process)}")

# --- Load CLIP model ---
processor = AutoProcessor.from_pretrained(model_name)
model = AutoModel.from_pretrained(model_name)
device = "cuda" if torch.cuda.is_available() else "cpu"
model.to(device)
print(f"Using CLIP model: {model_name} on device: {device}")

# --- Image fetch function ---
def fetch_image(index_row_tuple):
    index, row = index_row_tuple
    image_url = row["url"]
    caption = row["caption"]

    # start_time = time.time()

    try:
        response = requests.get(image_url, timeout=10)
        response.raise_for_status()
        image = Image.open(BytesIO(response.content)).convert("RGB")

        # duration = time.time() - start_time
        # print(f"[Downloaded] {image_url} in {duration:.2f}s")

        return (index, image_url, caption, image)
    except:
        return (index, image_url, caption, None)

# --- Main processing ---
new_embedding_results = []
total_processed = len(existing_embeddings_df)

for batch_start in tqdm(range(0, len(images_to_process), batch_size), desc="Processing Batches"):
    batch_end = min(batch_start + batch_size, len(images_to_process))
    batch_rows = list(images_to_process.iloc[batch_start:batch_end].iterrows())

    # Parallel image download
    st_time = time.time()
    with ThreadPoolExecutor(max_workers=number_of_workers) as executor:
        fetched_images = list(executor.map(fetch_image, batch_rows))
    elapsed_time = time.time() - st_time
    tqdm.write(f"Batch {batch_start // batch_size + 1} processed in {elapsed_time:.2f}s")

    valid_batch = [item for item in fetched_images if item[3] is not None]
    failed_batch = [item for item in fetched_images if item[3] is None]

    # Process valid images on GPU
    if valid_batch:
        indices, urls, captions, images = zip(*valid_batch)
        try:
            inputs = processor(images=list(images), return_tensors="pt", padding=True).to(device)
            with torch.no_grad():
                image_features = model.get_image_features(**inputs)
            image_embeddings = image_features / image_features.norm(p=2, dim=-1, keepdim=True)

            for i in range(len(valid_batch)):
                new_embedding_results.append({
                    'original_image_index': indices[i],
                    'url': urls[i],
                    'caption': captions[i],
                    'embeddings_result': image_embeddings[i].cpu().numpy(),
                    'similarity': None,
                    'parquet_file_name': parquet_file
                })
        except Exception as e:
            tqdm.write(f"⚠️ Batch error: {e}. Attempting to filter invalid images...")

            # Try each image individually to isolate faulty ones
            for i in range(len(valid_batch)):
                img = images[i]
                try:
                    single_input = processor(images=[img], return_tensors="pt", padding=True).to(device)
                    with torch.no_grad():
                        single_feature = model.get_image_features(**single_input)
                    single_embedding = single_feature / single_feature.norm(p=2, dim=-1, keepdim=True)

                    new_embedding_results.append({
                        'original_image_index': indices[i],
                        'url': urls[i],
                        'caption': captions[i],
                        'embeddings_result': single_embedding[0].cpu().numpy(),
                        'similarity': None,
                        'parquet_file_name': parquet_file
                    })
                except Exception as inner_e:
                    tqdm.write(f"❌ Skipping image at index {indices[i]} due to error: {inner_e}")
                    new_embedding_results.append({
                        'original_image_index': indices[i],
                        'url': urls[i],
                        'caption': captions[i],
                        'embeddings_result': None,
                        'similarity': None,
                        'parquet_file_name': parquet_file
                    })

    # Log failed downloads
    for idx, url, cap, _ in failed_batch:
        new_embedding_results.append({
            'original_image_index': idx,
            'url': url,
            'caption': cap,
            'embeddings_result': None,
            'similarity': None,
            'parquet_file_name': parquet_file
        })

    total_processed += len(fetched_images)

    # Checkpoint saving
    if total_processed % save_interval < batch_size or batch_end == len(images_to_process):
        df_new_batch = pd.DataFrame(new_embedding_results)
        combined_df = pd.concat([existing_embeddings_df, df_new_batch], ignore_index=True)
        combined_df.drop_duplicates(subset=['original_image_index'], keep='last', inplace=True)

        # test_path = os.path.join(output_dir_for_embeddings, f"{parquet_file.replace('.parquet', '')}_embeddings_{len(combined_df)}.parquet")
        # combined_df.to_parquet(test_path, index=False)
        # combined_df.to_parquet(output_filepath, index=False)

        table = pa.Table.from_pandas(combined_df)
        test_path = os.path.join(output_dir_for_embeddings, f"{parquet_file.replace('.parquet', '')}_embeddings_{len(combined_df)}.parquet")
        pq.write_table(table, test_path, row_group_size=row_grp_size)
        pq.write_table(table, output_filepath, row_group_size=row_grp_size)

        # --- Keep only the X most recent test parquet files ---
        # List all checkpoint files for this image set
        test_files = sorted(
            glob.glob(os.path.join(output_dir_for_embeddings, f"{parquet_file.replace('.parquet', '')}_embeddings_*.parquet")),
            key=os.path.getmtime,  # sort by modification time
            reverse=True
        )

        # Keep only the 2 most recent files
        for old_file in test_files[keep_n_recent_saves:]:
            try:
                os.remove(old_file)
                tqdm.write(f"🗑️ Deleted old checkpoint: {old_file}")
            except Exception as e:
                tqdm.write(f"⚠️ Could not delete {old_file}: {e}")

                tqdm.write(f"✅ Checkpoint saved: {len(combined_df)} rows (including failures)")

                existing_embeddings_df = combined_df.copy()
                new_embedding_results = []

# --- Final Save ---
print("Finalizing remaining embeddings...")
if new_embedding_results:
    df_final = pd.DataFrame(new_embedding_results)
    combined_df = pd.concat([existing_embeddings_df, df_final], ignore_index=True)
    combined_df.drop_duplicates(subset=['original_image_index'], keep='last', inplace=True)

    final_dir = output_filepath.replace('clip_embeddings_resumable', 'final_clip_embeddings')
    os.makedirs(os.path.dirname(final_dir), exist_ok=True)
    # combined_df.to_parquet(final_dir, index=False)
    table = pa.Table.from_pandas(combined_df)
    pq.write_table(table, final_dir, row_group_size=row_grp_size)

    tqdm.write(f"✅ Final save complete: {len(combined_df)} rows written to {final_dir}")

# --- Final Stats ---
print(f"\nEmbedding process complete for {model_name}")
final_df = pd.read_parquet(output_filepath)
success_count = len(final_df.dropna(subset=['embeddings_result']))
print(f"✅ Successful embeddings: {success_count}")
print(f"📦 Total attempted (incl. failed): {len(final_df)}")


# Make this continue to process from the largest index onwards so if you have to split the parquet file you can just continue from the last index processed also allows for a passsed index value from which to start processing
# from as an additional option 

# Look into loading only the last row_group to reduce on memory needed and see if you can append to a parquet file without having to read all the data in memory beforehand again this is to handle large datasets in the future

Loading DataFrame from dataset/relaion2B-en-research-safe\0000.parquet...
DataFrame loaded with 16388209 rows.
Filtered data: 16388209 rows. Limiting to first 3000000.
Resuming: Found 2090000 existing embeddings.
Remaining images to process: 910000
Using CLIP model: openai/clip-vit-large-patch14 on device: cuda


Processing Batches:   0%|          | 0/4550 [00:00<?, ?it/s]

Batch 1 processed in 48.29s


c:\MastersRepos\ARI5902-Research-Topics-in-AI\LAION-5B Testing\.venv\lib\site-packages\transformers\models\clip\modeling_clip.py:491: UserWarning: 1Torch was not compiled with flash attention. (Triggered internally at C:\actions-runner\_work\pytorch\pytorch\builder\windows\pytorch\aten\src\ATen\native\transformers\cuda\sdp_utils.cpp:555.)
  attn_output = torch.nn.functional.scaled_dot_product_attention(
c:\MastersRepos\ARI5902-Research-Topics-in-AI\LAION-5B Testing\.venv\lib\site-packages\PIL\Image.py:1056: UserWarning: Palette images with Transparency expressed in bytes should be converted to RGBA images
  warnings.warn(


Batch 2 processed in 159.07s
Batch 3 processed in 12.40s
Batch 4 processed in 28.33s
Batch 5 processed in 11.81s
Batch 6 processed in 35.45s
Batch 7 processed in 32.06s
Batch 8 processed in 11.44s
Batch 9 processed in 15.14s
Batch 10 processed in 11.35s
Batch 11 processed in 15.16s
Batch 12 processed in 21.95s
Batch 13 processed in 28.12s
Batch 14 processed in 15.18s
Batch 15 processed in 18.70s
Batch 16 processed in 11.46s
Batch 17 processed in 20.60s
Batch 18 processed in 23.04s
Batch 19 processed in 11.30s
Batch 20 processed in 11.20s
Batch 21 processed in 61.81s
Batch 22 processed in 62.26s
Batch 23 processed in 10.49s
Batch 24 processed in 10.45s
Batch 25 processed in 21.18s
🗑️ Deleted old checkpoint: clip_embeddings_resumable\all_images_openai_clip_vit_large_patch14\0000_embeddings_2085000.parquet
Batch 26 processed in 20.82s
Batch 27 processed in 11.35s
Batch 28 processed in 31.89s
Batch 29 processed in 13.29s
Batch 30 processed in 33.60s
Batch 31 processed in 38.12s
Batch 32 pr

In [ ]:
# View number of images without an embedding
import pandas as pd

# Load the saved Parquet file
df = pd.read_parquet(r"C:\MastersRepos\ARI5902-Research-Topics-in-AI\LAION-5B Testing\clip_embeddings_resumable\all_images_openai_clip_vit_large_patch14\0000_embeddings_1825000.parquet")

# Count rows with missing embeddings
missing_count = df['embeddings_result'].isnull().sum()

print(f"❌ Missing embeddings: {missing_count}")
print(f"✅ Total rows: {len(df)}")
print(f"📊 Success rate: {(1 - missing_count / len(df)) * 100:.2f}%")


❌ Missing embeddings: 613726
✅ Total rows: 1825000
📊 Success rate: 66.37%


In [ ]:
# import pandas as pd
# import requests
# from PIL import Image
# from io import BytesIO
# import torch
# from transformers import AutoProcessor, AutoModel
# import numpy as np
# import pyarrow as pa
# import pyarrow.parquet as pq
# import os
# import time
# from tqdm import tqdm
# from concurrent.futures import ThreadPoolExecutor

# # --- Config ---
# input_path = r"C:\MastersRepos\ARI5902-Research-Topics-in-AI\LAION-5B Testing\clip_embeddings_resumable\all_images_openai_clip_vit_large_patch14\0000_embeddings_1825000_PROCESSED_330000_retry_filled.parquet"
# save_path = r"C:\MastersRepos\ARI5902-Research-Topics-in-AI\LAION-5B Testing\clip_embeddings_resumable\all_images_openai_clip_vit_large_patch14\0000_embeddings_1825000.parquet"
# output_path = input_path
# clip_model_names = ["openai/clip-vit-base-patch16", "openai/clip-vit-base-patch32", "openai/clip-vit-large-patch14"]
# model_index = 2
# model_name = clip_model_names[model_index]
# device = "cuda" if torch.cuda.is_available() else "cpu"
# save_interval = 5000
# row_grp_size = 300000
# batch_size = number_of_workers = 200
# start_offset = 330000

# # --- Load Model & Processor ---
# processor = AutoProcessor.from_pretrained(model_name)
# model = AutoModel.from_pretrained(model_name).to(device)
# model.eval()

# # --- Load Data ---
# df = pd.read_parquet(output_path if os.path.exists(output_path) else input_path)
# print(f"🔍 Loaded {len(df)} rows")

# missing_df = df[df['embeddings_result'].isnull()].copy()
# print(f"❌ Retrying {len(missing_df)} missing embeddings...")

# old_output_file = None
# processed = 0
# successful_embeddings = 0

# # --- Image Fetch Function ---
# def fetch_image(index_row_tuple):
#     index, row = index_row_tuple
#     image_url = row["url"]
#     caption = row["caption"]
#     original_index = row["original_image_index"]
#     try:
#         response = requests.get(image_url, timeout=10)
#         response.raise_for_status()
#         image = Image.open(BytesIO(response.content)).convert("RGB")
#         return (original_index, image_url, caption, image)
#     except:
#         return (original_index, image_url, caption, None)

# # --- Reprocess in Batches ---
# for batch_start in tqdm(range(start_offset, len(missing_df), batch_size), desc="Reprocessing in Batches"):
#     batch_end = min(batch_start + batch_size, len(missing_df))
#     batch_rows = list(missing_df.iloc[batch_start:batch_end].iterrows())

#     # Parallel image fetching
#     with ThreadPoolExecutor(max_workers=number_of_workers) as executor:
#         fetched_images = list(executor.map(fetch_image, batch_rows))

#     valid_batch = [item for item in fetched_images if item[3] is not None]
#     failed_batch = [item for item in fetched_images if item[3] is None]

#     print(f"VALID BATCH SIZE: {len(valid_batch)} - INVALID BATCH SIZE: {len(failed_batch)}")

#     # Process valid images
#     if valid_batch:
#         indices, urls, captions, images = zip(*valid_batch)
#         try:
#             inputs = processor(images=list(images), return_tensors="pt", padding=True).to(device)
#             with torch.no_grad():
#                 image_features = model.get_image_features(**inputs)
#             image_embeddings = image_features / image_features.norm(p=2, dim=-1, keepdim=True)

#             for i in range(len(valid_batch)):
#                 df.loc[df["original_image_index"] == indices[i], "embeddings_result"] = [image_embeddings[i].cpu().numpy()]
#                 successful_embeddings += 1

#         except Exception as e:
#             tqdm.write(f"⚠️ Batch processing error: {e}. Trying images individually...")
#             for i in range(len(valid_batch)):
#                 try:
#                     single_input = processor(images=[images[i]], return_tensors="pt").to(device)
#                     with torch.no_grad():
#                         single_feature = model.get_image_features(**single_input)
#                     single_embedding = single_feature / single_feature.norm(p=2)
#                     df.loc[df["original_image_index"] == indices[i], "embeddings_result"] = [single_embedding.cpu().numpy()]
#                     successful_embeddings += 1
#                 except Exception as inner_e:
#                     pass
#                     # tqdm.write(f"❌ Failed image at index {indices[i]}: {inner_e}")

#     # Log failed ones (leave as NaN)
#     for original_index, url, caption, _ in failed_batch:
#         pass  # No update needed

#     processed += len(fetched_images)

#     # --- Save checkpoint ---
#     if processed % save_interval < batch_size or batch_end == len(missing_df):
#         table = pa.Table.from_pandas(df)
#         current_output_path = save_path.replace(".parquet", f"_PROCESSED_{processed}_retry_filled.parquet")

#         if old_output_file and os.path.exists(old_output_file):
#             os.remove(old_output_file)
#             tqdm.write(f"🗑️ Removed old checkpoint file: {old_output_file}")

#         pq.write_table(table, current_output_path, row_group_size=row_grp_size)
#         tqdm.write(f"💾 Saved checkpoint: {successful_embeddings}/{processed} images embedded")

#         old_output_file = current_output_path

# # --- Final Save ---
# table = pa.Table.from_pandas(df)
# pq.write_table(table, output_path, row_group_size=row_grp_size)
# print(f"✅ Final save complete to: {output_path}")


In [ ]:
# # convert to use row groups, something like this:

# import pyarrow as pa
# import pyarrow.parquet as pq

# # Convert Pandas DataFrame to PyArrow Table
# table = pa.Table.from_pandas(combined_df)

# # Write to Parquet with specific row group size
# pq.write_table(table, output_filepath, row_group_size=100_000)


Try downlaoding a subset of images only and save them to disk to determien how long it actually takes, then processs the image using CLIp and see which part is the actal throttel.


Issue seems to be realted to model execution time look into that

# LOOK INTO PARQUET FILE COMPRESSION

pq.write_table(table, output_path, compression='zstd', compression_level=10)